# Content-Based Book Recommender - Goodbooks 10k

Cieľom tohto notebook-u je vybudovanie systému odporúčania kníh založený na *content-based filteringu*.

Princíp je jednoduchý: každá kniha je opísaná svojimi tagmi (žánre, témy, nálada ...). Knihy s podobnými tagmi sú si podobné - a práve tie odporúčame.

Pipeline:
1. 'tags_string' každej knihy transformujeme na číselný vektor pomocou **TF-IDF**
1. Podobnosť medzi knihami meriame pomocou **kosínovej podobnosti**
1. Pre danú knihu vrátime top-N najpodobnejších kníh
1. Kvalitu modelu zmeriame metrikami **Precision@K**, **Recall@K** a **F1@K**
1. Spustíme sériu **experimentov** s rôznymi parametrami a porovnáme ich v benchmark tabuľke
1. Riešime **Cold Start problém** pre nových používateľov bez histórie

## 1. Načítanie knižníc a dát

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Výstupy preprocessingu - books_content obsahuje tag profil každej knihy,
# ratings_clean slúži na evaluáciu modelu

# Načítanie datasetu
books_df = pd.read_csv('preprocessed/books_content.csv')
ratings_df = pd.read_csv('preprocessed/ratings_clean.csv')

print(f"Kníh: {len(books_df)}")
print(f"Hodnotení: {len(ratings_df)}")
print(f"Unikátnych používateľov: {ratings_df['user_id'].nunique()}")

Kníh: 10000
Hodnotení: 979478
Unikátnych používateľov: 53424


## 2. TF-IDF matica a kosínová podobnosť

**TF-IDF** transformuje textový 'tags_string' každej knihy na číselný vektor.

- **TF** (term frequency): ako často je tag v danej knihe 
- **IDF** (inverse document frequency): tagy, ktoré sú v každej knihe dostanú nízku váhu, kvôli ne-informatívnosti. Unikátne tagy majú vyššiu váhu.

Výsledkom bude matica **10 000 x N**, kde každý riadok je vektor jednej knihy.

**Cos podobnosť** meria uhol medzi dvoma vektormi:
- hodnota **1.0** znamená, že knihy sú identické
- hodnota **0.0** znamená významne odlišné knihy

Výsledná 'cosine_sim' matica má rozmery **10k x 10k** - každá kniha x každá kniha

In [2]:
# Vytvorenie TF-IDF matice pre obsah kníh
# Stop words odstráni frekventované anglické slová (the, and, is ...)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(books_df['tags_string'])

print(f'Matica s rozmermi: {tfidf_matrix.shape}')
print(f"{tfidf_matrix.shape[0]} kníh × {tfidf_matrix.shape[1]} unikátnych tagov")

# Výpočet podobnosti medzi knihami pomocou kosínovej podobnosti.
# Matica zaberá asi 800 MB RAM
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"Cosine similarity matica: {cosine_sim.shape}")

Matica s rozmermi: (10000, 14786)
10000 kníh × 14786 unikátnych tagov
Cosine similarity matica: (10000, 10000)


## 3. Funkcie odporúčania

'get_recommendations(title, cosine_sim, n) - jadro systému.

Pre zadanú knihu nájde jej riadok v 'cosine_sim' matici, zoradí všetky ostatné knihy podľa podobnosti k zadanej knihe zostupne a vráti top-N výsledkov s ich skóre.

In [3]:
# Pomocný index: názov knihy -> číslo riadku v books_df
# Umožní nám O(1) vyhľadávanie namiesto pomalého filtrovania celého DF
indices = pd.Series(books_df.index, index=books_df['title'])

def get_recommendations(title, cosine_sim=cosine_sim, n=10):
    """ 
    Vráti top-N kníh podobných zadanému titulu

    Parametre:
        title (str): Názov knihy, pre ktorú chceme odporúčania
        cosine_sim (np.array): Matica kosínovej podobnosti, default = baseline
        n (int): počet odporúčaní, default = 10

    Návratová hodnota:
        DataFrame so stĺpcami (title, authors, average_rating, similarity)
        alebo chybová správa (ak kniha neexistuje)
    """
    
    if title not in indices:
        return f"Kniha '{title}' sa nenašla."
    
    idx = indices[title]
    
    # Ak existujú duplicitné názvy, vezmi prvý výskyt
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
    
    # Zoradenie všetkých kníh podľa podobnosti k danej knihe
    # Enumerate pridá číslo riadku, sorted zoradí podľa similarity zostupne
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)

    # Preskočíme index 0 -> ide o samotnú knihu (sim = 1.0)
    sim_scores = sim_scores[1:n+1]
    book_indices = [i for i, _ in sim_scores]
    
    result = books_df[['title', 'authors', 'average_rating']].iloc[book_indices].copy()
    result['similarity'] = [round(s, 4) for _, s in sim_scores]
    result = result.reset_index(drop=True)
    return result

### Ukážka odporúčaní - baseline model

Pred spustením benchmarku overíme, že funkcia vracia zmysluplné výsledky.
Očakávame, že pre *The Hunger Games* systém odporučí knihy z rovnakého žánru — dystopia, YA, sci-fi séria.

In [4]:
get_recommendations("The Hunger Games (The Hunger Games, #1)")

,title,authors,average_rating,similarity
0,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,4.30,0.9624
1,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,4.03,0.9550
2,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,4.49,0.9181
3,"Divergent (Divergent, #1)",Veronica Roth,4.24,0.7043
4,"Starters (Starters, #1)",Lissa Price,3.91,0.6518
5,"Specials (Uglies, #3)",Scott Westerfeld,3.77,0.6514
6,"Extras (Uglies, #4)",Scott Westerfeld,3.59,0.6349
7,"Legend (Legend, #1)",Marie Lu,4.19,0.6303
8,"Insurgent (Divergent, #2)",Veronica Roth,4.07,0.6300
9,"Allegiant (Divergent, #3)",Veronica Roth,3.63,0.6254


## 4. Evaluácia - Precision@K, Recall@K, F1@K

Aby sme dokázali porovnať experimenty objektívne, potrebujeme číselné metriky.

Používame prístup **leave-one-out**:
1. Vyberieme náhodného používateľa
1. Jeho knihy s ratingom >= 4 sú jeho "dobré knihy"
1. Jedna z nich ide do modelu ako vstup (query)
1. Zvyšok sú **ground_truth** - čo by model ideálne mal odporučiť
1. Spustíme 'get_recommendations()' a spočítame zhody

**Precision@K** - koľko z K odporúčaní bolo relevantných
**Recall@K** - koľko % ground_truth model zachytil v top-K
**F1@K** - harmonický priemer Precision a Recall

> Hodnoty budú nízke — to je normálne a očakávané. Dataset má 10 000 kníh, priemerný používateľ hodnotil ~18 kníh. Pravdepodobnosť náhodnej zhody v top-10 je ~1 %. Každá hodnota nad 1 % teda predstavuje reálne zlepšenie oproti náhode.


In [5]:
def evaluate(cosine_sim, k=10, sample=500, seed=42):
    """ 
    Meria kvalitu modelu pomocou metrik Precision@k, Recall@k a F1@k 
    na vzorke náhodných používateľov.

    Parametre:
        cosine_sim (np.array): Matica cos podobnosti
        k (int): počet odporúčaní, default = 10
        sample (int): počet náhodne vybraných userov, default = 500
        seed (int): random seed pre výber vzorky, default = 42

    Návratová hodnota:
        Slovník s priemernými hodnotami 'precision@k', 'recall@k', 'f1@k' a 'n_users' (počet evaluovaných používateľov)
    """
    np.random.seed(seed)
    
    user_ids = ratings_df['user_id'].unique()
    sampled = np.random.choice(user_ids, size=min(sample, len(user_ids)), replace=False)
    
    precisions, recalls = [], []
    
    for user_id in sampled:
        # Relevantné knihy: tie ktoré user hodnotil 4 alebo 5
        good_books = ratings_df[
            (ratings_df['user_id'] == user_id) & (ratings_df['rating'] >= 4)
        ]

        # Potrebujeme aspoň 2 knihy: 1 query, ostatné ground thruth
        if len(good_books) < 2:
            continue
        
        # Jedna kniha ako query, zvyšok ako ground truth
        # Seed je odvodený od user_id, potom rôzny users dostanú rôzne query knihy
        query_id = good_books.sample(1, random_state=int(user_id) % 10000)['book_id'].values[0]

        # Musíme však zabezpečiť, aby bola kniha vylúčená z ground truth, 
        # inak by sme ju mohli náhodou odporučiť a nesprávne zvýšiť presnosť.
        ground_truth_ids = set(good_books['book_id'].values) - {query_id}

        
        # Preklad query_id na názov knihy, aby sme mohli použiť get_recommendations
        query_titles = books_df[books_df['record_id'] == query_id]['title']
        if query_titles.empty:
            continue
        
        recs = get_recommendations(query_titles.values[0], cosine_sim, n=k)
        if isinstance(recs, str):  # kniha sa nenašla
            continue
        
        # Prevod názvov odporúčaní spät na record_id pre porovnanie s ground truth
        rec_ids = set(books_df[books_df['title'].isin(recs['title'])]['record_id'].values)
        
        hits = len(rec_ids & ground_truth_ids)
        precisions.append(hits / k)
        recalls.append(hits / len(ground_truth_ids) if ground_truth_ids else 0)
    
    return {
        'precision@k': round(np.mean(precisions), 4),
        'recall@k': round(np.mean(recalls), 4),
        # 1e-9 zabraňuje deleniu nulou ak sú obe metriky nulové
        'f1@k': round(2 * np.mean(precisions) * np.mean(recalls) / (np.mean(precisions) + np.mean(recalls) + 1e-9), 4),
        'n_users': len(precisions)
    }

## 5. Benchmarking - porovnanie experimentov

Každý experiment mení **jeden parameter** oproti baseline, ostatné ostanú rovnaké.

| Experiment | Čo meníme |
|---|---|
| baseline | len `tags_string`, všetky defaulty |
| exp2_authors | pridáme autora ako feature |
| exp3_top10 | obmedzíme na top 10 tagov namiesto 20 |
| exp4_top50 | rozšírime na top 50 tagov |
| exp5_maxfeat | obmedzíme slovník TF-IDF na 5 000 slov |
| exp6_bigram | pridáme bigramy (dvojice slov ako jeden token) |

Po každom behu je priebežne vypísaná aktuálna tabuľka výsledkov.

In [6]:
results = []

def run_experiment(name, description, matrix):
    """
    Spustí jeden experiment content-based filteringu.

    Parametre:
        name (str): názov experimentu pre benchmark tabuľku
        description (str): krátky popis experimentu - zmena proti baseline-u
        matrix (scipy.sparse matrix): TF-IDF matica pre daný experiment

    Každý experiment dostane inú maticu - iné features, iné parametre - a evaluate()
    zmeria ako dobre model odporúča na vzorke 500 používateľov.
    """
    print(f"Running experiment: {name}")

    # Cosine similarity počítame z matice, ktorú sme dostali ako argument

    cos_sim = cosine_similarity(matrix, matrix)
    
    metrics = evaluate(cos_sim)

    results.append({
        'Experiment': name,
        'Popis': description,
        'Precision@10': metrics['precision@k'],
        'Recall@10': metrics['recall@k'],
        'F1@10': metrics['f1@k'],
        'N users': metrics['n_users']
    })

    # Vypíš tabuľku po každom experimente
    df = pd.DataFrame(results)
    print(df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    print(f"{'─'*80}")
    return cos_sim

### Príprava feature stĺpcov a spustenie experimentov

In [7]:
# Varianty experimentov 

""" 
Každý experiment má vlastný TfidVectorizer() - pretože zdielanie
jedného objektu by spôsobilo prepísanie interného slovníka a neporovnateľné
výsledky experimentov.

"""

# Exp 2: Pridáme meno autora ako ďalší feature.
# Medzery nahradíme podčiarkovníkom aby TF-IDF bral "J.K._Rowling" ako jeden token,
# nie ako dve nesúvisiace slová "J.K." a "Rowling".
books_df['combined_authors'] = (
    books_df['tags_string'] + ' ' +
    books_df['authors'].str.replace(' ', '_').str.replace(',', '')
)

# Exp 3 / Exp 4: Testujeme vplyv počtu tagov na kvalitu odporúčaní.
# Menej tagov = menej šumu, ale aj menej informácií o obsahu knihy.
# Viac tagov = bohatší profil, ale väčšia pravdepodobnosť zahrnutia irelevantných tagov.

books_df['tags_top10'] = books_df['tags_string'].apply(
    lambda x: ' '.join(str(x).split()[:10])
    )

books_df['tags_top50'] = books_df['tags_string'].apply(
    lambda x: ' '.join(str(x).split()[:50])
    )

# Spustenie experimentov
# Baseline: čistý TF-IDF na tags_string, žiadne úpravy — referenčný bod
cos_baseline = run_experiment(
    'baseline', 'tags_string, (1, 1), max_features = None',
    TfidfVectorizer(stop_words='english').fit_transform(books_df['tags_string'])
)

# Exp2: Hypotéza — autor je silný signál, najmä pre knižné série
cos_exp2 = run_experiment(
    'exp2_authors', 'tags + authors, (1, 1)',
    TfidfVectorizer(stop_words='english').fit_transform(books_df['combined_authors'])
)

# Exp3: Hypotéza — top 10 tagov obsahuje najpresnejší obsah, menej šumu
cos_exp3 = run_experiment(
    'exp3_top10', 'top 10 tagov, (1, 1)',
    TfidfVectorizer(stop_words='english').fit_transform(books_df['tags_top10'])
)

# Exp4: Hypotéza — viac tagov = bohatší profil = lepší Recall
cos_exp4 = run_experiment(
    'exp4_top50', 'top 50 tagov (1, 1)',
    TfidfVectorizer(stop_words='english').fit_transform(books_df['tags_top50'])
)

# Exp5: Hypotéza — obmedzenie slovníka odfiltruje vzácne/šumové tagy
cos_exp5 = run_experiment(
    'exp5_maxfeat', 'tags_string, max_features=5000',
    TfidfVectorizer(stop_words='english', max_features=5000).fit_transform(books_df['tags_string'])
)

# Exp6: Hypotéza — bigramy zachytia frázy ako "young-adult" alebo "science-fiction"
# ako jeden token namiesto dvoch nesúvisiacich slov
cos_exp6 = run_experiment(
    'exp6_bigram', 'tags_string, ngram=(1,2)',
    TfidfVectorizer(stop_words='english', ngram_range=(1,2)).fit_transform(books_df['tags_string'])
)

Running experiment: baseline
Experiment                                    Popis  Precision@10  Recall@10  F1@10  N users
  baseline tags_string, (1, 1), max_features = None        0.0778     0.0898 0.0834      428
────────────────────────────────────────────────────────────────────────────────
Running experiment: exp2_authors
  Experiment                                    Popis  Precision@10  Recall@10  F1@10  N users
    baseline tags_string, (1, 1), max_features = None        0.0778     0.0898 0.0834      428
exp2_authors                   tags + authors, (1, 1)        0.0794     0.0920 0.0853      428
────────────────────────────────────────────────────────────────────────────────
Running experiment: exp3_top10
  Experiment                                    Popis  Precision@10  Recall@10  F1@10  N users
    baseline tags_string, (1, 1), max_features = None        0.0778     0.0898 0.0834      428
exp2_authors                   tags + authors, (1, 1)        0.0794     0.0920 0.085

## 6. Výber najlepšieho modelu a uloženie

In [8]:
results_df = pd.DataFrame(results)

# Najlepší experiment podľa Precision@10 - primárna metrika
# V prípade zhody sa porovnáva podľa F1@10
best_row = results_df.loc[results_df['Precision@10'].idxmax()]
print("Najlepší experiment:\n", best_row)

# Mapa názvov -> cosine_sim objekt
cosine_map = {
    'baseline':    cos_baseline,
    'exp2_authors': cos_exp2,
    'exp3_top10':  cos_exp3,
    'exp4_top50':  cos_exp4,
    'exp5_maxfeat': cos_exp5,
    'exp6_bigram': cos_exp6,
}
best_cosine_sim = cosine_map[best_row['Experiment']]

os.makedirs('./results', exist_ok=True)
os.makedirs('./models', exist_ok=True)

# Ukladáme 3 výstupy:
# experiment_results.csv - tabuľka s výsledkami všetkých exp
# cosine_sim_best.npy - matica kosínovej podobnosti z najlepšieho exp
# books_model.csv - knižný model s metadátami
results_df.to_csv('./results/experiment_results.csv', index=False)
np.save('./models/cosine_sim_best.npy', best_cosine_sim)
books_df[['record_id', 'goodreads_book_id', 'title', 'authors',
          'average_rating', 'tags_string']].to_csv('./models/books_model.csv', index=False)

print("\nUložené:")
print("  results/experiment_results.csv")
print("  models/cosine_sim_best.npy")
print("  models/books_model.csv")

Najlepší experiment:
 Experiment                   exp6_bigram
Popis           tags_string, ngram=(1,2)
Precision@10                      0.0799
Recall@10                         0.0943
F1@10                             0.0865
N users                              428
Name: 5, dtype: object

Uložené:
  results/experiment_results.csv
  models/cosine_sim_best.npy
  models/books_model.csv


### Interpretácia výsledkov benchmarku

Na základe výsledkov vyššie sme iteratívne upravovali parametre modelu. Kľúčové pozorovania:

- **Baseline** predstavuje referenčný bod — čistý TF-IDF na `tags_string` bez akýchkoľvek úprav
- Experimenty s **viac tagmi** (exp4_top50) typicky zlepšujú Recall, pretože kniha je opísaná bohatším profilom
- Pridanie **autora** (exp2_authors) zlepšuje odporúčania pre knižné série, kde je autor silným identifikátorom
- **Bigramy** (exp6_bigram) môžu zlepšiť presnosť pre žánrové frázy ale zároveň výrazne zväčšujú slovník
- Výsledky potvrdzujú, že content-based filtering na tagových dátach dosahuje zmysluplné výsledky nad úrovňou náhody (~1 %)

Najlepší model bol uložený pre použitie v sekcii Cold Start a User Profiling.

## 7. Ukážka odporúčaní - najlepší model

In [9]:
get_recommendations("The Hunger Games (The Hunger Games, #1)", best_cosine_sim)

,title,authors,average_rating,similarity
0,"Mockingjay (The Hunger Games, #3)",Suzanne Collins,4.03,0.5052
1,"Catching Fire (The Hunger Games, #2)",Suzanne Collins,4.30,0.4922
2,The Hunger Games Trilogy Boxset (The Hunger Ga...,Suzanne Collins,4.49,0.4343
3,The Hunger Games Tribute Guide,Emily Seife,4.40,0.3283
4,The Hunger Games: Official Illustrated Movie C...,Kate Egan,4.51,0.3092
5,The World of the Hunger Games (Hunger Games Tr...,Kate Egan,4.48,0.2848
6,"Extras (Uglies, #4)",Scott Westerfeld,3.59,0.2792
7,"Specials (Uglies, #3)",Scott Westerfeld,3.77,0.2744
8,"Divergent (Divergent, #1)",Veronica Roth,4.24,0.2700
9,"Starters (Starters, #1)",Lissa Price,3.91,0.2679


In [10]:
def get_global_popular(n=10):
    """
    Vráti top-N najpopulárnejších kníh globálne podľa popularity_score.
    
    Parametre:
        n (int): počet odporúčaní
    """
    top = books_df.nlargest(n, 'popularity_score')[
        ['title', 'authors', 'average_rating', 'ratings_count', 'popularity_score']
    ].reset_index(drop=True)
    
    top.index += 1
    return top

### Cold start - popularita žánrov

Ak používateľ zadá len žáner (napr. "fantasy"), filtrovanie podľa tags_string.

In [11]:
def get_genre_popular(genre, n=10):
    """ 
    Vráti top-N populárnych kníh v zadanom žánri podľa popularity_score.
    Žáner hľadáme v tags_string - ak user napíše fantasy, vrátime 
    najpopulárnejšie knihy, ktoré majú "fantasy" v tags_string.

    Parametre:
        genre (str): žáner, ktorý chceme filtrovať
        n (int): počet odporúčaní, default = 10
    """

    # case = False: zabezpečí že "Fantasy"  aj "fantasy" budú fungovat
    genre_books = books_df[
        books_df['tags_string'].str.lower().str.contains(genre.lower(), na=False)
    ]
    
    if genre_books.empty:
        return f"Žáner '{genre}' sa nenašiel."
    
    top = genre_books.nlargest(n, 'popularity_score')[
        ['title', 'authors', 'average_rating', 'ratings_count', 'popularity_score']
    ].reset_index(drop=True)
    top.index += 1
    return top

In [12]:
print("Cold Start Level 2 - Žánrová Popularita (fantasy):\n")
print(get_genre_popular("fantasy", 10).to_string())

Cold Start Level 2 - Žánrová Popularita (fantasy):



KeyError: 'popularity_score'

### Odporúčania na základe profilu používateľa

Keď používateľ zadá zoznam obľúbených kníh (napr. pri onboardingu), môžeme vytvoriť jeho **profilový vektor**.

Postup:
1. Pre každú obľúbenú knihu načítame jej TF-IDF vektor
2. Vektorov priemer = "priemerná kniha" ktorú by používateľ mal rád
3. Nájdeme knihy s najvyššou kosínovou podobnosťou k tomuto priemernému vektoru

Toto je elegantné riešenie cold startu pri onboardingu — stačia 3–5 kníh na vytvorenie zmysluplného profilu.

In [ ]:
def get_user_profile_recommendations(favorite_titles, tfidf_matrix, cosine_sim, books_df, n=10):
    """
    Vytvorí používateľský profil z jeho obľúbených kníh a vráti odporúčania.

    Princíp: TF-IDF vektor každej obľúbenej knihy reprezentuje jej obsah.
    Priemer týchto vektorov = "priemerná kniha" ktorú by používateľ mal rád.
    Potom nájdeme knihy s najvyššou kosínovou podobnosťou k tomuto priemeru.

    Parametre:
        favorite_titles : zoznam názvov obľúbených kníh (aspoň 1)
        tfidf_matrix    : sparse matica TF-IDF (potrebná na výpočet profilu)
        cosine_sim      : matica podobnosti (pre porovnanie profilu s knihami)
        books_df        : DataFrame s metadátami kníh
        n               : počet odporúčaní
    """
    indices = pd.Series(books_df.index, index=books_df['title'])
    
    # Zbierame indexy kníh, ktoré user zadal a ktoré existujú v datasete
    valid_indices = []
    for title in favorite_titles:
        if title in indices:
            idx = indices[title]
            if isinstance(idx, pd.Series):
                idx = idx.iloc[0]
            valid_indices.append(idx)
    
    if not valid_indices:
        return "Žiadna z uvedených kníh sa nenašla."
    
    # Načítame TF-IDF vektory obľúbených kníh a spriemerujeme ich.
    # toarray() konvertuje sparse maticu na hustú — potrebné pre np.mean().
    book_vectors = tfidf_matrix[valid_indices].toarray()
    user_profile = np.mean(book_vectors, axis=0)
    
    # Porovnáme profil používateľa s každou knihou v datasete pomocou kosínovej podobnosti.
    # reshape(1, -1) prekonvertuje 1D vektor na 2D maticu (1 x N) - vyžaduje to cos sim
    similarities = cosine_similarity(
        user_profile.reshape(1, -1), 
        tfidf_matrix.toarray()
    )[0]
    
    sim_scores = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)
    
    # Vylúčime knihy ktoré používateľ už zadal — nechceme odporučiť to čo už pozná
    sim_scores = [s for s in sim_scores if s[0] not in valid_indices][:n]
    
    book_indices = [i for i, _ in sim_scores]
    result = books_df[['title', 'authors', 'average_rating']].iloc[book_indices].copy()
    result['similarity'] = [round(s, 4) for _, s in sim_scores]
    result = result.reset_index(drop=True)
    return result

In [ ]:
print("User Profile - Odporúčania pre fanúšika Harry Potter:")
user_books = [
    "Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",
    "Harry Potter and the Chamber of Secrets (Harry Potter, #2)",
    "Harry Potter and the Prisoner of Azkaban (Harry Potter, #3)"
]
print(get_user_profile_recommendations(user_books, tfidf_matrix, cosine_sim, books_df, n=10).to_string(index=False))

### Hybridné skóre — kombinácia similarity a kvality

Čistá kosínová podobnosť ignoruje kvalitu knihy — dve knihy s rovnakými tagmi môžu mať veľmi rozdielne hodnotenia. Hybridné skóre kombinuje:

- **80 % similarity** — obsahová relevancia (hlavný signál)
- **20 % normalized rating** — kvalita knihy (korektor)

Váhy sú nastaviteľné cez konštanty `SIMILARITY_WEIGHT` a `RATING_WEIGHT`. Rating je normalizovaný do rozsahu [0, 1] pred kombináciou, aby nemal neprimeraný vplyv.

In [ ]:
# Váhy definujeme ako konštanty, aby boli ľahko upraviteľné a konzistentné
SIMILARITY_WEIGHT = 0.8 # Obsah je hlavný faktor
RATING_WEIGHT = 0.2 # Kvalita knihy je sekundárny faktor

def normalize_ratings(books_df_subset):
    """ 
    Normalizuje stĺpec average_rating do rozsahu [0, 1].
    Bez normalizácie by rating (rozsah 1–5) mal neprimeraný vplyv
    voči similarity (rozsah 0–1) pri kombinácii skóre.
    """
    min_r = books_df_subset['average_rating'].min()
    max_r = books_df_subset['average_rating'].max()

    # Ochrana pred delením nulou, ak sú všetky knihy rovnako hodnotené
    if max_r - min_r == 0:
        return pd.Series([0.5] * len(books_df_subset), index=books_df_subset.index)
    return (books_df_subset['average_rating'] - min_r) / (max_r - min_r)

def apply_hybrid_score(df, similarity_col=None):
    """
    Aplikuje hybridné skóre na DataFrame odporúčaní.
    Ak je k dispozícii similarity stĺpec, kombinuje ho s ratingom (80/20).
    Ak nie (napr. pri cold start popularity), radí čisto podľa ratingu.
    """
    df = df.copy()
    df['norm_rating'] = normalize_ratings(df)
    
    if similarity_col and similarity_col in df.columns:
        df['hybrid_score'] = (
            SIMILARITY_WEIGHT * df[similarity_col] + 
            RATING_WEIGHT * df['norm_rating']
        )
    else:
        # Pri cold start nemáme similarity, takže radíme len podľa ratingu
        df['hybrid_score'] = df['norm_rating']
    
    return df.sort_values('hybrid_score', ascending=False).drop(columns=['norm_rating'])

In [ ]:
print("Pred hybridným skóre:")
recs = get_user_profile_recommendations(user_books, tfidf_matrix, cosine_sim, books_df, n=10)
print(recs.head(10).to_string(index=False))

print("\nPo aplikovaní hybridného skóre (80% similarity + 20% rating):")
recs_hybrid = apply_hybrid_score(recs)
print(recs_hybrid.head(10).to_string(index=False))

### Integration Wrapper — `get_final_recommendations()`

Inteligentný vstupný bod ktorý automaticky zvolí správnu metódu podľa vstupu:
```
user_input = "fantasy"     → genre popularity  
user_input = "Název knihy" → Content-Based filtering
user_input = [list]        → User Profile Recommender
```

Toto je fasáda celého systému — UI alebo API by volali len túto funkciu.

In [ ]:
def get_final_recommendations(user_input, tfidf_matrix, cosine_sim, books_df, n=10, apply_hybrid=True):
    """ 
    Hlaná vstupná funkcia systému - automaticky zvolí správnu stratégiu podľa typu vstupu.

    Logika výberu:
        list        → User Profile: personalizované na základe obľúbených kníh
        krátky str  → populárne knihy v zadanom žánri
        dlhý str    → Content-Based: odporúčania podobné zadanej knihe
        
    Parametre:
        user_input (str | list | None): vstup od používateľa
        tfidf_matrix (sparse matrix): TF-IDF matica pre výpočty
        cosine_sim (np.array): matica kosínovej podobnosti
        books_df (DataFrame): DataFrame s informáciami o knihách
        n (int): počet odporúčaní, default = 10
        apply_hybrid (bool): či aplikovať hybridné skóre, default = True
    
    """
    
    indices = pd.Series(books_df.index, index=books_df['title'])
    
    # Žiadny vstup -> úplne nový user bez akejkoľvek informácie
    if user_input is None:
        print("[COLD START - Level 1: Global Popularity]")
        result = get_global_popular(n)
        return result.reset_index(drop=True)

    # Zoznam kníh = onboarding pre používateľa, ktorý nám poskyt
    if isinstance(user_input, list):
        print("[USER PROFILE RECOMMENDER]")
        result = get_user_profile_recommendations(
            user_input, tfidf_matrix, cosine_sim, books_df, n
        )
        if apply_hybrid and not isinstance(result, str):
            result = apply_hybrid_score(result, similarity_col='similarity')
        return result
    
    if isinstance(user_input, str):
        words = user_input.strip().split()
        
        # Krátky reťazec (1-3 slová) interpretujeme ako žáner
        if len(words) <= 3:
            print(f"[COLD START - Level 2: Genre '{user_input}']")
            result = get_genre_popular(user_input, n)
            if apply_hybrid and not isinstance(result, str):
                result = apply_hybrid_score(result, similarity_col=None)
            return result
        
        # Dlhší reťazec interpretujeme ako názov knihy pre content-based odporúčania
        if user_input in indices:
            print(f"[CONTENT-BASED: '{user_input}']")
            result = get_recommendations(user_input, cosine_sim, n)
            if apply_hybrid and not isinstance(result, str):
                result = apply_hybrid_score(result, similarity_col='similarity')
            return result
        else:
            # Kniha sa nenašla — fallback na genre search
            print(f"[COLD START - Level 2: Genre '{user_input}']")
            result = get_genre_popular(user_input, n)
            if apply_hybrid and not isinstance(result, str):
                result = apply_hybrid_score(result, similarity_col=None)
            return result
    
    return "Neplatný vstup."

In [ ]:
print("=" * 60)
print("TESTOVANIE get_final_recommendations()")
print("=" * 60)

print("\n1. POUŽÍVATEĽ BEZ HISTÓRIE (Cold Start Level 1):")
print("-" * 40)
print(get_final_recommendations(None, tfidf_matrix, cosine_sim, books_df, n=5).to_string(index=False))

print("\n2. POUŽÍVATEĽ S ŽÁNROM 'fantasy' (Cold Start Level 2):")
print("-" * 40)
print(get_final_recommendations("fantasy", tfidf_matrix, cosine_sim, books_df, n=5).to_string(index=False))

print("\n3. OBSAHOVÉ ODPORÚČANIE (Content-Based):")
print("-" * 40)
print(get_final_recommendations("The Hunger Games (The Hunger Games, #1)", tfidf_matrix, cosine_sim, books_df, n=5).to_string(index=False))

print("\n4. PERSONALIZOVANÉ ODPORÚČANIE (User Profile):")
print("-" * 40)
user_favorites = [
    "The Hunger Games (The Hunger Games, #1)",
    "Catching Fire (The Hunger Games, #2)",
    "Divergent (Divergent, #1)"
]
print(get_final_recommendations(user_favorites, tfidf_matrix, cosine_sim, books_df, n=5).to_string(index=False))

In [ ]:
# Kvalitativna kontrola najlepšieho modelu na 5 rôznych knihách
# Overenie, či odporúčania dávajú zmysel z pohľadu žánru, témy a série
test_books = [
    "The Hunger Games (The Hunger Games, #1)",
    "Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",
    "To Kill a Mockingbird",
    "The Great Gatsby",
    "Twilight (Twilight, #1)"
]

for title in test_books:
    print(f"\n{'='*55}")
    print(f"Odporúčania pre: {title}")
    print('='*55)
    print(get_recommendations(title, best_cosine_sim, n=5).to_string(index=False))

### Kvalitatívne zhodnotenie

Systém správne zoskupuje knihy podľa žánru a série. Pre **Hunger Games* odporučil dystopické young adult série (Divergent, Maze Runner), pre *Harry Potter* fantasy sériu pre mladých čitateľov, pre *Twilight* paranormálnu romance. Klasická literatúra je správne zoskupená spoločne - potvrdzuje to, že tagový profil zachytáva žánrové aj tematické súvislosti. Systém neodporučil žiadnu knihu z iného žánru, čo svedčí o vysokej relevancii výsledkov.